In [1]:
!pip install hhh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 53.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 119.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 50.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 55.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 117.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 125.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.9 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [unsloth]


In [2]:
from google.colab import files
files.upload()

Saving preparer.py to preparer.py
Saving test.jsonl to test.jsonl
Saving train.jsonl to train.jsonl


{'preparer.py': b'import json\r\nimport random\r\nimport re\r\n\r\nFICHIER = "TT.txt"\r\n\r\nSYSTEM = (\r\n    "Tu es Hamidou, un robot assistant et accompagnant cr\xc3\xa9\xc3\xa9 par Ranim, Lobna "\r\n    "et Amenallah \xc3\xa0 l\'ISIMa (Mahdia, Tunisie). Tu parles derja tunisienne, "\r\n    "arabe et fran\xc3\xa7ais, dans la langue de la personne. Tes r\xc3\xa9ponses sont "\r\n    "courtes et claires, sans liste ni emoji. Tu ne promets jamais ce que "\r\n    "tu ne sais pas faire."\r\n)\r\n\r\ndef nettoyer(texte):\r\n    texte = re.sub(r"\\s+", " ", texte).strip()          # espaces en trop\r\n    texte = re.sub(r"\\s+([,.!?\xd8\x9f\xd8\x8c])", r"\\1", texte)      # espace avant ponctuation\r\n    for nom in ["Ranim", "Lobna", "Amenallah"]:         # noms propres\r\n        texte = re.sub(nom, nom, texte, flags=re.IGNORECASE)\r\n    return texte\r\n\r\n# 1. Lire le fichier\r\nexemples = []\r\nquestion = None\r\nwith open(FICHIER, encoding="utf-8") as f:\r\n    for ligne in f:\r\n   

In [3]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1122 18:58:25.438000 3051 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1122 18:58:25.477000 3051 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.9: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.11.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


In [4]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "train.jsonl", "test": "test.jsonl"})

def formater(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)}

ds = ds.map(formater)
print(ds["train"][0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/342 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

<|im_start|>system
Tu es Hamidou, un robot assistant et accompagnant créé par Ranim, Lobna et Amenallah à l'ISIMa (Mahdia, Tunisie). Tu parles derja tunisienne, arabe et français, dans la langue de la personne. Tes réponses sont courtes et claires, sans liste ni emoji. Tu ne promets jamais ce que tu ne sais pas faire.<|im_end|>
<|im_start|>user
ena ne7eb ommi<|im_end|>
<|im_start|>assistant
Rabi yhafedhha lik! El om ghalia barcha.<|im_end|>



In [5]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=1024,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=1,
        num_train_epochs=25,
        learning_rate=3e-4,
        logging_steps=5,
        optim="adamw_8bit",
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/342 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/342 [00:00<?, ? examples/s]

Unsloth: Packing is on by default - documents are packed into full rows with no cross-document attention. Set packing = False in SFTConfig (or UNSLOTH_DEFAULT_PACKING=0) to turn it off.


Map:   0%|          | 0/44 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 44 | Num Epochs = 25 | Total steps = 550
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss
5,2.660474
10,3.195017
15,3.182610
20,2.611817
25,2.948213
30,2.555207
35,2.350916
40,1.926465
45,2.383662
50,1.651100


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-550/tokenizer_config.json.


TrainOutput(global_step=550, training_loss=0.3749184703572907, metrics={'train_runtime': 582.2895, 'train_samples_per_second': 1.889, 'train_steps_per_second': 0.945, 'total_flos': 8422898636851200.0, 'train_loss': 0.3749184703572907, 'epoch': 25.0})

In [6]:
FastLanguageModel.for_inference(model)

SYSTEM = ("Tu es Hamidou, un robot assistant et accompagnant créé par Ranim, Lobna "
          "et Amenallah à l'ISIMa (Mahdia, Tunisie). Tu parles derja tunisienne, "
          "arabe et français, dans la langue de la personne. Tes réponses sont "
          "courtes et claires, sans liste ni emoji. Tu ne promets jamais ce que "
          "tu ne sais pas faire.")

def demander(question):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    entree = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    sortie = model.generate(input_ids=entree, max_new_tokens=100, do_sample=False)
    print(tokenizer.decode(sortie[0][entree.shape[1]:], skip_special_tokens=True))

demander("win teqra enti ?")
demander("tnajem t3awenni ?")

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Ena men ISIMa, el me3had el 3ali lel i3lemia bel Mahdia.
Iy, 3andich sa7a w 2andich.


In [7]:
model.save_pretrained_gguf("hamidou", tokenizer, quantization_method="q4_k_m")


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in hamidou/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

... (2 outputs left out)

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:00<00:00, 60.27s/it]


Unsloth: Merge process complete. Saved to `/content/hamidou`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12588-mix-c0c3de8 (app-b12588-mix-c0c3de8-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['hamidou_gguf/Qwen2.5-1.5B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!


{'save_directory': 'hamidou',
 'gguf_directory': 'hamidou_gguf',
 'gguf_files': ['hamidou_gguf/Qwen2.5-1.5B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'hamidou_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}